In [3]:
import sys
!{sys.executable} -m pip install nltk pandas scikit-learn numpy

## 1. Imports & Setup
> Load all required libraries and download NLTK resources.

In [4]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## 2. Load Data
> Load resume and job description CSV files into pandas DataFrames.

In [5]:
df_resumes = pd.read_csv('data/UpdatedResumeDataSet.csv')
df_jobs = pd.read_csv('data/job_title_des.csv')

df_jobs.drop(columns=['Unnamed: 0'], inplace=True)

print("Resumes:", df_resumes.shape)
print("Jobs:", df_jobs.shape)

Resumes: (962, 2)
Jobs: (2277, 2)


## 3. Text Cleaning
> Define and apply the clean_text function to both resumes and job descriptions.

In [6]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

df_resumes['cleaned'] = df_resumes['Resume'].apply(clean_text)
df_jobs['cleaned'] = df_jobs['Job Description'].apply(clean_text)

print("✅ Cleaning done!")
print(df_resumes.columns.tolist())
print(df_jobs.columns.tolist())

✅ Cleaning done!
['Category', 'Resume', 'cleaned']
['Job Title', 'Job Description', 'cleaned']


In [7]:
print(df_resumes.columns.tolist())
print(df_jobs.columns.tolist())

['Category', 'Resume', 'cleaned']
['Job Title', 'Job Description', 'cleaned']


## 4. Save Cleaned Data
> Export the cleaned resume and job description DataFrames to CSV files.

In [8]:
df_resumes.to_csv('data/cleaned_resumes.csv', index=False)
df_jobs.to_csv('data/cleaned_jobs.csv', index=False)

print("✅ Saved cleaned_resumes.csv")
print("✅ Saved cleaned_jobs.csv")

✅ Saved cleaned_resumes.csv
✅ Saved cleaned_jobs.csv


## 5. TF-IDF Vectorization
> Fit TF-IDF on combined cleaned text and transform resumes and jobs into vectors.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

# Fit on both resumes and job descriptions combined
all_text = list(df_resumes['cleaned']) + list(df_jobs['cleaned'])
tfidf.fit(all_text)

# Transform each separately
resume_vectors = tfidf.transform(df_resumes['cleaned'])
job_vectors = tfidf.transform(df_jobs['cleaned'])

print("✅ TF-IDF done!")
print("Resume matrix shape:", resume_vectors.shape)
print("Job matrix shape:", job_vectors.shape)

✅ TF-IDF done!
Resume matrix shape: (962, 5000)
Job matrix shape: (2277, 5000)


## 6. Cosine Similarity Matching
> Define a resume-to-job similarity function and test it on a sample pair.

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

def match_resume_to_job(resume_text, job_index):
    cleaned_resume = clean_text(resume_text)
    resume_vec = tfidf.transform([cleaned_resume])
    job_vec = job_vectors[job_index]
    
    score = cosine_similarity(resume_vec, job_vec)[0][0]
    return round(score * 100, 2)

# Test it
sample_resume = df_resumes['Resume'][0]
sample_job = df_jobs['Job Title'][0]
score = match_resume_to_job(sample_resume, 0)

print(f"Resume Category: {df_resumes['Category'][0]}")
print(f"Job Title: {sample_job}")
print(f"Match Score: {score}%")

Resume Category: Data Science
Job Title: Flutter Developer
Match Score: 0.53%


## 7. Top Job Matches
> Retrieve and display the top 5 most similar jobs for a given resume.

In [11]:
def get_top_matches(resume_text, top_n=5):
    cleaned_resume = clean_text(resume_text)
    resume_vec = tfidf.transform([cleaned_resume])
    
    scores = cosine_similarity(resume_vec, job_vectors)[0]
    top_indices = scores.argsort()[::-1][:top_n]
    
    results = []
    for i in top_indices:
        results.append({
            'job_title': df_jobs['Job Title'][i],
            'score': round(scores[i] * 100, 2)
        })
    
    return results

# Test it
sample_resume = df_resumes['Resume'][0]
top_matches = get_top_matches(sample_resume)

print(f"Resume Category: {df_resumes['Category'][0]}")
print("\nTop 5 Matching Jobs:")
for match in top_matches:
    print(f"  {match['job_title']} — {match['score']}%")

Resume Category: Data Science

Top 5 Matching Jobs:
  Machine Learning — 26.96%
  Machine Learning — 25.45%
  Machine Learning — 24.21%
  Machine Learning — 24.08%
  Machine Learning — 23.19%


## 8. Deduplicate Jobs
> Remove duplicate job titles and recompute job vectors for unique roles.

In [12]:
# Drop duplicate job titles, keep first occurrence
df_jobs_unique = df_jobs.drop_duplicates(subset='Job Title').reset_index(drop=True)

# Re-vectorize with deduplicated jobs
job_vectors_unique = tfidf.transform(df_jobs_unique['cleaned'])

def get_top_matches_unique(resume_text, top_n=5):
    cleaned_resume = clean_text(resume_text)
    resume_vec = tfidf.transform([cleaned_resume])
    
    scores = cosine_similarity(resume_vec, job_vectors_unique)[0]
    top_indices = scores.argsort()[::-1][:top_n]
    
    results = []
    for i in top_indices:
        results.append({
            'job_title': df_jobs_unique['Job Title'][i],
            'score': round(scores[i] * 100, 2)
        })
    
    return results

# Test again
top_matches = get_top_matches_unique(df_resumes['Resume'][0])

print(f"Resume Category: {df_resumes['Category'][0]}")
print("\nTop 5 Matching Jobs:")
for match in top_matches:
    print(f"  {match['job_title']} — {match['score']}%")

Resume Category: Data Science

Top 5 Matching Jobs:
  Machine Learning — 15.15%
  Software Engineer — 9.25%
  Java Developer — 8.02%
  Full Stack Developer — 7.0%
  Backend Developer — 6.62%


## 9. Save Model
> Save the trained TF-IDF vectorizer and job vectors to the models folder.

In [13]:
import pickle

# Save tfidf vectorizer
with open('models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Save job vectors
with open('models/job_vectors.pkl', 'wb') as f:
    pickle.dump(job_vectors_unique, f)

# Save deduplicated jobs dataframe
df_jobs_unique.to_csv('data/cleaned_jobs_unique.csv', index=False)

print("✅ tfidf_vectorizer.pkl saved")
print("✅ job_vectors.pkl saved")
print("✅ cleaned_jobs_unique.csv saved")

✅ tfidf_vectorizer.pkl saved
✅ job_vectors.pkl saved
✅ cleaned_jobs_unique.csv saved


## 10. Category Mappings
> Define ground-truth relevant job titles for each resume category.

In [14]:
# Mapping resume categories to relevant job titles in our dataset
category_to_jobs = {
    'Data Science': ['Machine Learning', 'Data Scientist', 'Data Analyst', 'Business Analyst'],
    'Java Developer': ['Java Developer', 'Software Engineer', 'Full Stack Developer', 'Backend Developer'],
    'Python Developer': ['Python Developer', 'Django Developer', 'Machine Learning', 'Software Engineer'],
    'Web Designing': ['Web Designer', 'Full Stack Developer', 'Frontend Developer', 'UI/UX Designer'],
    'DevOps Engineer': ['DevOps Engineer', 'Cloud Engineer', 'Software Engineer'],
    'Testing': ['Software Tester', 'QA Engineer', 'Automation Testing', 'Software Engineer'],
    'HR': ['HR Manager', 'HR Executive', 'Recruiter'],
    'Hadoop': ['Hadoop Developer', 'Data Engineer', 'Big Data Engineer', 'Machine Learning'],
    'Blockchain': ['Blockchain Developer', 'Software Engineer', 'Full Stack Developer'],
    'ETL Developer': ['ETL Developer', 'Data Engineer', 'Database Administrator'],
    'Operations Manager': ['Operations Manager', 'Project Manager', 'Business Analyst'],
    'Sales': ['Sales Manager', 'Business Development', 'Marketing Manager'],
    'Mechanical Engineer': ['Mechanical Engineer', 'Design Engineer', 'Production Engineer'],
    'Arts': ['Graphic Designer', 'Web Designer', 'UI/UX Designer'],
    'Database': ['Database Administrator', 'ETL Developer', 'Data Engineer'],
    'Health and fitness': ['Health Coach', 'Fitness Trainer', 'Nutritionist'],
    'Electrical Engineering': ['Electrical Engineer', 'Electronics Engineer'],
    'PMO': ['Project Manager', 'Operations Manager', 'Business Analyst'],
    'Business Analyst': ['Business Analyst', 'Data Analyst', 'Operations Manager'],
    'DotNet Developer': ['DotNet Developer', 'Software Engineer', 'Full Stack Developer'],
    'Automation Testing': ['Automation Testing', 'QA Engineer', 'Software Tester'],
    'Network Security Engineer': ['Network Engineer', 'Security Engineer', 'DevOps Engineer'],
    'Civil Engineer': ['Civil Engineer', 'Structural Engineer', 'Project Manager'],
    'SAP Developer': ['SAP Developer', 'ERP Consultant', 'Software Engineer'],
    'Advocate': ['Legal Advisor', 'Advocate', 'Compliance Officer'],
}

print("✅ Category mappings defined!")
print(f"Total categories mapped: {len(category_to_jobs)}")

✅ Category mappings defined!
Total categories mapped: 25


## 11. Evaluation - Precision@5 and MRR
> Compute Precision@5 and MRR across resumes using category relevance mappings.

In [15]:
def get_top_matches_eval(resume_text, top_n=5):
    cleaned_resume = clean_text(resume_text)
    resume_vec = tfidf.transform([cleaned_resume])
    scores = cosine_similarity(resume_vec, job_vectors_unique)[0]
    top_indices = scores.argsort()[::-1][:top_n]
    return [df_jobs_unique['Job Title'][i] for i in top_indices]

precision_scores = []
mrr_scores = []

for _, row in df_resumes.iterrows():
    category = row['Category']
    resume = row['Resume']
    
    relevant_jobs = category_to_jobs.get(category, [])
    if not relevant_jobs:
        continue
    
    top_matches = get_top_matches_eval(resume, top_n=5)
    
    # Precision@5
    hits = sum(1 for job in top_matches if job in relevant_jobs)
    precision_scores.append(hits / 5)
    
    # MRR
    mrr = 0
    for rank, job in enumerate(top_matches, start=1):
        if job in relevant_jobs:
            mrr = 1 / rank
            break
    mrr_scores.append(mrr)

avg_precision = sum(precision_scores) / len(precision_scores)
avg_mrr = sum(mrr_scores) / len(mrr_scores)

print("📊 Evaluation Results")
print("=" * 35)
print(f"  Precision@5 : {avg_precision:.4f} ({avg_precision*100:.2f}%)")
print(f"  MRR         : {avg_mrr:.4f} ({avg_mrr*100:.2f}%)")
print(f"  Total Evaluated: {len(precision_scores)} resumes")

📊 Evaluation Results
  Precision@5 : 0.1249 (12.49%)
  MRR         : 0.2759 (27.59%)
  Total Evaluated: 962 resumes


## 12. Per Category Breakdown
> Report Precision@5 and MRR scores grouped by resume category.

In [16]:
from collections import defaultdict

category_precision = defaultdict(list)
category_mrr = defaultdict(list)

for _, row in df_resumes.iterrows():
    category = row['Category']
    resume = row['Resume']
    
    relevant_jobs = category_to_jobs.get(category, [])
    if not relevant_jobs:
        continue
    
    top_matches = get_top_matches_eval(resume, top_n=5)
    
    hits = sum(1 for job in top_matches if job in relevant_jobs)
    category_precision[category].append(hits / 5)
    
    mrr = 0
    for rank, job in enumerate(top_matches, start=1):
        if job in relevant_jobs:
            mrr = 1 / rank
            break
    category_mrr[category].append(mrr)

print(f"{'Category':<30} {'Precision@5':>12} {'MRR':>10}")
print("=" * 55)
for cat in sorted(category_precision.keys()):
    p = sum(category_precision[cat]) / len(category_precision[cat])
    m = sum(category_mrr[cat]) / len(category_mrr[cat])
    print(f"{cat:<30} {p*100:>11.2f}% {m*100:>9.2f}%")

Category                        Precision@5        MRR
Advocate                              0.00%      0.00%
Arts                                  0.00%      0.00%
Automation Testing                    0.00%      0.00%
Blockchain                           16.00%     31.67%
Business Analyst                      0.00%      0.00%
Civil Engineer                        0.00%      0.00%
Data Science                         20.00%     95.00%
Database                             18.18%     65.91%
DevOps Engineer                      18.91%     63.64%
DotNet Developer                     17.14%     25.24%
ETL Developer                         8.00%      9.00%
Electrical Engineering                0.00%      0.00%
HR                                    0.00%      0.00%
Hadoop                               20.00%     75.00%
Health and fitness                    0.00%      0.00%
Java Developer                       47.14%     78.57%
Mechanical Engineer                   0.00%      0.00%
Network Se

## 13. Filtered Evaluation
> Evaluate metrics only for resume categories represented in the jobs dataset.

In [17]:
# Only evaluate categories that have matching jobs in our dataset
valid_categories = [
    'Data Science', 'Java Developer', 'Python Developer', 
    'Web Designing', 'DevOps Engineer', 'Hadoop', 'Blockchain',
    'ETL Developer', 'Database', 'DotNet Developer', 
    'Automation Testing', 'Network Security Engineer', 'Testing'
]

filtered_resumes = df_resumes[df_resumes['Category'].isin(valid_categories)]

precision_scores_f = []
mrr_scores_f = []

for _, row in filtered_resumes.iterrows():
    category = row['Category']
    resume = row['Resume']
    
    relevant_jobs = category_to_jobs.get(category, [])
    if not relevant_jobs:
        continue
    
    top_matches = get_top_matches_eval(resume, top_n=5)
    
    hits = sum(1 for job in top_matches if job in relevant_jobs)
    precision_scores_f.append(hits / 5)
    
    mrr = 0
    for rank, job in enumerate(top_matches, start=1):
        if job in relevant_jobs:
            mrr = 1 / rank
            break
    mrr_scores_f.append(mrr)

avg_precision_f = sum(precision_scores_f) / len(precision_scores_f)
avg_mrr_f = sum(mrr_scores_f) / len(mrr_scores_f)

print("📊 Filtered Evaluation Results (relevant categories only)")
print("=" * 50)
print(f"  Precision@5 : {avg_precision_f:.4f} ({avg_precision_f*100:.2f}%)")
print(f"  MRR         : {avg_mrr_f:.4f} ({avg_mrr_f*100:.2f}%)")
print(f"  Total Evaluated: {len(precision_scores_f)} resumes")

📊 Filtered Evaluation Results (relevant categories only)
  Precision@5 : 0.2087 (20.87%)
  MRR         : 0.4608 (46.08%)
  Total Evaluated: 576 resumes
